# Allianz 파인튜닝 데이터셋 생성

## 개요
- `rag_utils.py`의 `generate_answer()` 활용해 질문-답변 쌍 생성
- 목표: **400개** JSONL 데이터셋
- 파인튜닝 목적: 출처 형식 고정 / 면책 고지 자동 삽입 / 추천 거절 / 개인정보(식별 가능한 경우만) 차단

## 카테고리 비율 (회의 결정 사항)
| 카테고리 | 비율 | 수량(400기준) | 설명 |
|---|---|---|---|
| 면책 고지 | 35% | ~140개 | 일반 보험 정보 제공 + 면책 고지 포함 |
| 추천 방지 | 20% | ~80개 | 플랜 추천 요청 → 거절 |
| 문맥 | 10% | ~40개 | 복잡한 질문, 문맥 파악 필요 |
| 개인정보 차단 | 35% | ~140개 | 식별 가능 개인정보 포함 시만 차단 |

## 개인정보 차단 기준 (회의 결정)
- 차단 X: `"내가 당뇨인데 Allianz에서 커버되나요?"` → 진단명만 있는 경우 일반 답변
- 차단 O: `"제 보험번호 AL-12345이고 당뇨 진단인데"` → 개인 식별 정보 포함 시 차단

## 출처 표기 형식 (회의 결정)
- PDF: `[출처]: 파일명(괄호 제거) 연도 p.N`
  - ex. `DOC-Care-IBG-EN-1125 2025 p.3`
- 공식 사이트 참조 시: `[출처]: Allianz Care https://www.allianzcare.com/`

## 수량 줄이는 법 (나중에 줄여야 할 경우)
- 400→200개: 각 카테고리 리스트를 절반으로 슬라이싱
  예) `disclaimer_questions[:70]`, `reject_questions[:40]`, ...
- 400→100개: 각 카테고리를 1/4로
  예) `disclaimer_questions[:35]`, `reject_questions[:20]`, ...

## 실행 전 필요 조건
> `allianz_rag_utils.py`가 같은 경로에 있어야 함
> `vectordb/allianz/` 폴더에 Chroma DB가 구축되어 있어야 함
> (`python src/allianz/allianz_ingest.py --reload`로 사전 구축 필요)

---
## Cell 0 · 환경 로드

In [10]:
import os
import sys
import re
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

# rag_utils.py 경로 추가
# 이 노트북이 src/allianz/ 폴더에 있는 경우
BASE_DIR = Path('../../').resolve()  # Insurance_Benefit_Chatbot 루트
sys.path.insert(0, str(BASE_DIR / 'src' / 'allianz'))

from rag_utils import generate_answer, run_chat_turn

print('✅ rag_utils 로드 완료')
print(f'BASE_DIR: {BASE_DIR}')

✅ rag_utils 로드 완료
BASE_DIR: C:\SKN_24\3차 프로젝트(팀)


In [11]:
# 언어 감지 함수 (거절 패턴 답변 언어 분기용)
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text): return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text): return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text): return 'ko'
    return 'en'

def clean_source_name(filename: str) -> str:
    """출처 파일명 정제:
    - 괄호와 괄호 안 내용 제거: DOC-Care-IBG-EN-1125_개인고객용혜택가이드 -> DOC-Care-IBG-EN-1125
    - 언더스코어 뒤 한국어 설명 제거
    - 확장자 제거
    """
    name = filename
    name = re.sub(r'\s*\([^)]*\)', '', name)    # 소괄호 + 내용 제거
    name = re.sub(r'\s*\[[^\]]*\]', '', name)    # 대괄호 + 내용 제거
    name = re.sub(r'_(가-힣]+.*$)', '', name)     # 언더스코어 + 한국어 설명 제거
    name = re.sub(r'\.(pdf|csv|PDF|CSV)$', '', name)  # 확장자 제거
    return name.strip()

print('✅ 유틸 함수 정의 완료')

✅ 유틸 함수 정의 완료


---
## Cell 1 · 질문 목록 정의

### 카테고리 비율
- 면책고지(일반 보험 정보): 35% → 140개
- 추천 방지: 20% → 80개
- 문맥: 10% → 40개
- 개인정보 차단: 35% → 140개

### 수량 줄이는 법
- 400→200개: `disclaimer_questions[:70]`, `reject_questions[:40]`, `context_questions[:20]`, `privacy_block_questions[:70]`
- 400→100개: `disclaimer_questions[:35]`, `reject_questions[:20]`, `context_questions[:10]`, `privacy_block_questions[:35]`

In [12]:
# 카테고리별 질문 목록
# (question, type, category)
# type: 'rag' | 'reject_recommend' | 'reject_privacy'

# 면책 고지 (일반 보험 정보) 140개 
# Coverage, Cost, Eligibility, Preauth, Claim 포함
# 답변 마지막에 면책 고지 + 출처 자동 삽입 학습 목적
disclaimer_questions = [
    # Coverage — 글로벌 공통 (30개)
    ('Allianz Care에서 입원 치료(Inpatient treatment)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('What does Allianz Care cover for outpatient medical treatment?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 응급 치료(Emergency treatment)는 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care cover mental health treatment?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 암 치료(Cancer treatment)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('What maternity benefits does Allianz Care provide?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 치과 치료(Dental treatment)는 보장이 되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care cover vision care and optical treatment?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 물리치료(Physiotherapy)는 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('What prescription drug benefits does Allianz Care offer?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 심리 치료(Psychotherapy)는 보장이 되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care cover preventive care and health check-ups?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 재활 치료(Rehabilitation)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('What are the coverage limits for inpatient treatment under Allianz Care?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 만성 질환(Chronic condition) 치료는 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care cover home nursing care?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 장기 입원 시 보장 한도는 얼마인가요?', 'rag', 'disclaimer_coverage'),
    ('What alternative treatments does Allianz Care cover?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 출산 전후(Maternity) 검사 비용은 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care cover organ transplants?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 HIV/AIDS 치료는 보장이 되나요?', 'rag', 'disclaimer_coverage'),
    ('What does Allianz Care cover for psychiatric inpatient treatment?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 수술 후 합병증 치료는 보장이 되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care have any coverage for congenital conditions?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care의 Care Base, Care Enhanced, Care Signature 플랜 간 보장 차이는 무엇인가요?', 'rag', 'disclaimer_coverage'),
    ('What is NOT covered under Allianz Care (exclusions)?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 사고로 인한 응급 치료는 바로 보장이 되나요?', 'rag', 'disclaimer_coverage'),
    ('Does Allianz Care cover experimental or investigational treatments?', 'rag', 'disclaimer_coverage'),
    ('Allianz Care에서 체외수정(IVF) 같은 불임 치료는 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('What does Allianz Care cover for newborn care?', 'rag', 'disclaimer_coverage'),

    # Coverage — 지역별 (30개)
    ('싱가포르에서 Allianz Care 입원 치료 보장 한도는 얼마인가요?', 'rag', 'disclaimer_regional'),
    ('What are the outpatient benefits under Allianz Care in the UK?', 'rag', 'disclaimer_regional'),
    ('홍콩에서 Allianz Care 외래 진료 보장이 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('Does Allianz Care cover maternity in the UAE (Dubai)?', 'rag', 'disclaimer_regional'),
    ('베트남에서 Allianz Care로 응급 치료 받으면 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('What dental benefits does Allianz Care provide in France?', 'rag', 'disclaimer_regional'),
    ('인도네시아에서 Allianz Care 입원 보장은 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('Does Allianz Care cover specialist visits in Switzerland?', 'rag', 'disclaimer_regional'),
    ('중국에서 Allianz Care 외래 치료 보장 한도는 얼마인가요?', 'rag', 'disclaimer_regional'),
    ('What maternity benefits does Allianz Care offer in Lebanon?', 'rag', 'disclaimer_regional'),
    ('남미에서 Allianz Care 응급 치료는 어떻게 보장되나요?', 'rag', 'disclaimer_regional'),
    ('Is mental health treatment covered under Allianz Care in Singapore?', 'rag', 'disclaimer_regional'),
    ('영국에서 Allianz Care로 치과 치료 받을 수 있나요?', 'rag', 'disclaimer_regional'),
    ('What are the coverage limits for cancer treatment under Allianz Care in Hong Kong?', 'rag', 'disclaimer_regional'),
    ('프랑스에서 Allianz Care 입원 치료 절차는 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('Does Allianz Care cover physiotherapy in Indonesia?', 'rag', 'disclaimer_regional'),
    ('스위스에서 Allianz Care 약품 보장은 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('What emergency services does Allianz Care cover in Vietnam?', 'rag', 'disclaimer_regional'),
    ('두바이에서 Allianz Care 임신 관련 보장 내용은 무엇인가요?', 'rag', 'disclaimer_regional'),
    ('Does Allianz Care have a network hospital in China?', 'rag', 'disclaimer_regional'),
    ('레바논에서 Allianz Care 입원 사전승인은 어떻게 하나요?', 'rag', 'disclaimer_regional'),
    ('What is the annual benefit limit for outpatient care in Latin America under Allianz?', 'rag', 'disclaimer_regional'),
    ('싱가포르에서 Allianz Care 정신건강 치료 보장은 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('Does Allianz Care cover rehabilitation treatment in the UK?', 'rag', 'disclaimer_regional'),
    ('홍콩에서 Allianz Care 사전승인(Pre-authorisation) 절차는 무엇인가요?', 'rag', 'disclaimer_regional'),
    ('What dental services are covered under Allianz Care in the UAE?', 'rag', 'disclaimer_regional'),
    ('베트남에서 Allianz Care 처방약 보장은 어떻게 되나요?', 'rag', 'disclaimer_regional'),
    ('Is preventive care covered under Allianz Care in France?', 'rag', 'disclaimer_regional'),
    ('인도네시아에서 Allianz Care 치과 치료 보장 한도는 얼마인가요?', 'rag', 'disclaimer_regional'),
    ('Does Allianz Care cover mental health inpatient in Switzerland?', 'rag', 'disclaimer_regional'),

    # Pre-auth & Claim (30개)
    ('Allianz Care에서 입원 전 사전승인(Pre-authorisation)은 어떻게 신청하나요?', 'rag', 'disclaimer_preauth'),
    ('What happens if I do not get pre-authorisation for planned hospitalisation with Allianz?', 'rag', 'disclaimer_preauth'),
    ('Allianz Care 사전승인 없이 응급 입원한 경우 어떻게 처리되나요?', 'rag', 'disclaimer_preauth'),
    ('How long does pre-authorisation approval take at Allianz Care?', 'rag', 'disclaimer_preauth'),
    ('Allianz Care 사전승인 신청서(FRM-PreAuth)에 어떤 내용을 작성해야 하나요?', 'rag', 'disclaimer_preauth'),
    ('Does Allianz Care require pre-authorisation for outpatient specialist visits?', 'rag', 'disclaimer_preauth'),
    ('Allianz Care에서 사전승인이 필요한 치료 항목은 무엇인가요?', 'rag', 'disclaimer_preauth'),
    ('How do I submit a claim to Allianz Care after treatment?', 'rag', 'disclaimer_claim'),
    ('Allianz Care 보험금 청구 시 어떤 서류가 필요한가요?', 'rag', 'disclaimer_claim'),
    ('What is the claims submission deadline for Allianz Care?', 'rag', 'disclaimer_claim'),
    ('Allianz Care에서 청구가 거절되면 어떻게 하나요?', 'rag', 'disclaimer_claim'),
    ('Can I submit a claim to Allianz Care online?', 'rag', 'disclaimer_claim'),
    ('Allianz Care의 직접 청구(Direct billing)는 어떻게 작동하나요?', 'rag', 'disclaimer_claim'),
    ('What currencies does Allianz Care reimburse claims in?', 'rag', 'disclaimer_claim'),
    ('Allianz Care에서 해외 치료 후 청구 시 영수증 언어 제한이 있나요?', 'rag', 'disclaimer_claim'),
    ('How long does it take for Allianz Care to process a claim?', 'rag', 'disclaimer_claim'),
    ('Allianz Care 보험금 청구서(FRM-PCF)에 필수 기재 항목은 무엇인가요?', 'rag', 'disclaimer_claim'),
    ('Does Allianz Care cover claims for treatment received before enrollment?', 'rag', 'disclaimer_claim'),
    ('Allianz Care에서 긴급 해외 치료 후 청구 기한은 얼마인가요?', 'rag', 'disclaimer_claim'),
    ('What is the process for appealing a denied claim with Allianz Care?', 'rag', 'disclaimer_claim'),

    # Benefit limits & Cost (25개)
    ('Allianz Care Base 플랜의 연간 보장 한도는 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('What is the annual benefit limit for Allianz Care Enhanced plan?', 'rag', 'disclaimer_cost'),
    ('Allianz Care Signature 플랜의 입원 보장 한도는 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('Does Allianz Care have a deductible? How does it work?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 공동부담금(Co-payment)은 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('What is the out-of-pocket maximum under Allianz Care?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 치과 보장 한도는 연간 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('How much does Allianz Care cover for maternity treatment?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 정신건강 치료 연간 한도는 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('What is the benefit limit for outpatient treatment in Allianz Care Base?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 물리치료 세션당 보장 금액은 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('Does Allianz Care have a co-insurance requirement?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 처방약(Prescription drugs) 보장 한도는 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('What is the benefit for private room accommodation under Allianz Care?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 응급 치료 보장 한도는 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('How much does Allianz Care cover for cancer treatment annually?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 재활 치료 보장 한도는 얼마인가요?', 'rag', 'disclaimer_cost'),
    ('What is the optical benefit limit under Allianz Care?', 'rag', 'disclaimer_cost'),
    ('Allianz Care에서 심장 수술 비용 보장은 어떻게 되나요?', 'rag', 'disclaimer_cost'),
    ('How are benefit limits calculated under Allianz Care — per claim or per year?', 'rag', 'disclaimer_cost'),

    # Eligibility & General (25개)
    ('Allianz Care에 가입할 수 있는 자격 조건은 무엇인가요?', 'rag', 'disclaimer_eligibility'),
    ('Who can be enrolled as a dependent under Allianz Care?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 대기 기간(Waiting period)이 있는 항목은 무엇인가요?', 'rag', 'disclaimer_eligibility'),
    ('Does Allianz Care cover pre-existing conditions?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care의 보험 적용 지역(Area of cover)은 어떻게 되나요?', 'rag', 'disclaimer_eligibility'),
    ('What is the maximum age for enrollment in Allianz Care?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 플랜 업그레이드는 언제 할 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('How do I add a newborn to my Allianz Care policy?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 가입 후 취소(Cancellation) 조건은 어떻게 되나요?', 'rag', 'disclaimer_eligibility'),
    ('Does Allianz Care provide coverage worldwide or only in specific regions?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 보험 갱신(Renewal) 시 보험료가 오를 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('What documentation is required to enroll in Allianz Care?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 임신 중에 가입할 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('Is there a grace period for premium payment under Allianz Care?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 보험 시작일 이전에 발생한 치료비는 청구 가능한가요?', 'rag', 'disclaimer_eligibility'),
    ('Does Allianz Care offer individual and family plans?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 지역(Area of cover) 변경은 어떻게 신청하나요?', 'rag', 'disclaimer_eligibility'),
    ('What is the policy renewal process for Allianz Care?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 기존 질환(Pre-existing condition) 신고를 안 했으면 어떻게 되나요?', 'rag', 'disclaimer_eligibility'),
    ('How do I contact Allianz Care for claims support?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care 가입 시 건강 설문지(Medical questionnaire) 작성이 필요한가요?', 'rag', 'disclaimer_eligibility'),
    ('What languages does Allianz Care customer support operate in?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 보험료 납부 방법은 어떻게 되나요?', 'rag', 'disclaimer_eligibility'),
    ('Is there a mobile app or online portal for Allianz Care policyholders?', 'rag', 'disclaimer_eligibility'),
    ('Allianz Care에서 보험증권(Policy document)은 어디서 받을 수 있나요?', 'rag', 'disclaimer_eligibility'),
]

# 추천 방지 — 80개 
# 플랜 추천 요청 → 정보 비교만 제공, 추천/권유 거절
reject_recommend_questions = [
    ('Allianz Care Base랑 Enhanced 중 어떤 게 더 낫나요?', 'reject_recommend', 'reject_recommend'),
    ('저한테 맞는 Allianz Care 플랜 추천해주세요.', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan should I choose for cancer coverage?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care Signature가 Enhanced보다 훨씬 나은가요?', 'reject_recommend', 'reject_recommend'),
    ('싱가포르에서 Allianz Care 어떤 플랜이 제일 적합한지 알려주세요.', 'reject_recommend', 'reject_recommend'),
    ('비용 대비 가장 좋은 Allianz Care 플랜이 뭔가요?', 'reject_recommend', 'reject_recommend'),
    ('I want the most comprehensive Allianz Care plan. What do you recommend?', 'reject_recommend', 'reject_recommend'),
    ('가족이 있는데 Allianz Care 어떤 플랜이 좋나요?', 'reject_recommend', 'reject_recommend'),
    ('I want the cheapest Allianz Care option with good coverage. Which one?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 플랜 중 정신건강 커버가 제일 좋은 게 어떤 건가요?', 'reject_recommend', 'reject_recommend'),
    ('영국에서 Allianz Care 어떤 플랜 써야 해요?', 'reject_recommend', 'reject_recommend'),
    ('홍콩 거주자한테 어떤 Allianz 플랜이 적합한지 추천해주세요.', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan covers the most overseas services?', 'reject_recommend', 'reject_recommend'),
    ('약 많이 먹는데 어떤 Allianz 플랜이 유리해요?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care Base랑 Signature 중 뭐가 더 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('Should I upgrade from Allianz Care Base to Enhanced?', 'reject_recommend', 'reject_recommend'),
    ('임신 중인데 Allianz Care 어떤 플랜이 좋을까요?', 'reject_recommend', 'reject_recommend'),
    ('What Allianz Care plan is best for a family with young children?', 'reject_recommend', 'reject_recommend'),
    ('입원 잦을 것 같은데 Allianz 어떤 플랜이 유리해요?', 'reject_recommend', 'reject_recommend'),
    ('Is Allianz Care Signature worth the premium?', 'reject_recommend', 'reject_recommend'),
    ('치과 치료 많이 받는데 어떤 Allianz 플랜이 유리한가요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan should I pick if I want the lowest out-of-pocket costs with Allianz?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 플랜 중 가장 저렴한 건 뭐예요?', 'reject_recommend', 'reject_recommend'),
    ('What is the most cost-effective Allianz Care plan for expats?', 'reject_recommend', 'reject_recommend'),
    ('베트남에서 일하는데 Allianz Care 어떤 플랜이 맞아요?', 'reject_recommend', 'reject_recommend'),
    ('I need the best mental health coverage with Allianz. Which plan?', 'reject_recommend', 'reject_recommend'),
    ('어떤 Allianz 플랜이 해외 응급 치료를 가장 잘 커버하나요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan is best for retirees living abroad?', 'reject_recommend', 'reject_recommend'),
    ('Allianz 플랜 중 연간 비용이 가장 적게 드는 건 뭐예요?', 'reject_recommend', 'reject_recommend'),
    ('Should I go with Care Base or Enhanced for a young healthy person?', 'reject_recommend', 'reject_recommend'),
    ('임신 계획 중인데 Allianz 어떤 플랜 들어야 해요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan has the best maternity benefits?', 'reject_recommend', 'reject_recommend'),
    ('Allianz 연간 비용이 가장 적게 드는 플랜이 뭐예요?', 'reject_recommend', 'reject_recommend'),
    ('가입비 없는 Allianz 플랜 있나요? 그게 더 좋은 건가요?', 'reject_recommend', 'reject_recommend'),
    ('Should I enroll in Allianz Care Base or just go with Enhanced?', 'reject_recommend', 'reject_recommend'),
    ('약값 아끼려면 어떤 Allianz 플랜 써야 해요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan gives the most benefits for the lowest premium?', 'reject_recommend', 'reject_recommend'),
    ('두바이에서 정신건강 치료 받을 건데 어떤 Allianz 플랜이 나을까요?', 'reject_recommend', 'reject_recommend'),
    ('Should my spouse enroll in Allianz Care Base or Enhanced?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan has the best pharmacy benefits?', 'reject_recommend', 'reject_recommend'),
    ('Can you recommend the best Allianz option for someone with a chronic illness?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care Signature 가입할 가치 있나요?', 'reject_recommend', 'reject_recommend'),
    ('어떤 Allianz 플랜이 청구 절차가 제일 간단해요?', 'reject_recommend', 'reject_recommend'),
    ('What Allianz Care plan covers orthodontic treatment?', 'reject_recommend', 'reject_recommend'),
    ('Allianz 처음 가입하는 사람한테 어떤 플랜 알려줘요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan is best if I travel frequently between countries?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care Enhanced랑 Signature 비교해서 좋은 거 골라줘요.', 'reject_recommend', 'reject_recommend'),
    ('어떤 Allianz 플랜이 입원 보장이 제일 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('I want comprehensive cancer coverage. Which Allianz plan is best?', 'reject_recommend', 'reject_recommend'),
    ('중국 거주 외국인인데 Allianz 어떤 플랜 써야 해요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan is best for active duty military families?', 'reject_recommend', 'reject_recommend'),
    ('처방약 많이 먹는데 Allianz 어떤 플랜이 유리해요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan gives the most maternity benefits?', 'reject_recommend', 'reject_recommend'),
    ('알리안츠 케어 중 암 치료 보장이 제일 좋은 플랜이 뭐예요?', 'reject_recommend', 'reject_recommend'),
    ('Should I get Care Base now and upgrade later?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 플랜 중 응급 치료 보장이 가장 좋은 건 무엇인가요?', 'reject_recommend', 'reject_recommend'),
    ('Is there an Allianz Care plan that covers everything with no copay?', 'reject_recommend', 'reject_recommend'),
    ('Which plan minimizes out-of-pocket for inpatient mental health with Allianz?', 'reject_recommend', 'reject_recommend'),
    ('Allianz 처음 선택하는 건데 어떤 플랜이 제일 무난해요?', 'reject_recommend', 'reject_recommend'),
    ('I need surgery abroad. Which Allianz Care plan is best?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 플랜 비교해서 제일 좋은 플랜 골라줘요.', 'reject_recommend', 'reject_recommend'),
    ('스위스에 사는데 Allianz Care 어떤 플랜이 제일 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz plan covers the most services for expat families?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 가입 전 제일 유리한 플랜이 뭔지 알고 싶어요.', 'reject_recommend', 'reject_recommend'),
    ('I want the best value Allianz plan for my family.', 'reject_recommend', 'reject_recommend'),
    ('어떤 Allianz Care 플랜이 치과 보장이 제일 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan has the lowest deductible?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 중 해외 청구가 제일 쉬운 플랜 알려주세요.', 'reject_recommend', 'reject_recommend'),
    ('Can you help me pick the best Allianz plan based on my health needs?', 'reject_recommend', 'reject_recommend'),
    ('Which Allianz Care plan is better for families living in Southeast Asia?', 'reject_recommend', 'reject_recommend'),
    ('아이 있는 가정에 Allianz Care 어떤 플랜 추천해줘요?', 'reject_recommend', 'reject_recommend'),
    ('What Allianz Care plan has the lowest enrollment fee?', 'reject_recommend', 'reject_recommend'),
    ('Allianz Care 비교해서 가장 합리적인 플랜 알려주세요.', 'reject_recommend', 'reject_recommend'),
    ('Is Allianz Care Enhanced or Signature better for mental health coverage?', 'reject_recommend', 'reject_recommend'),
    ('Allianz 가입하려는데 제일 유리한 플랜이 뭔지 알고 싶어요.', 'reject_recommend', 'reject_recommend'),
    ('인도네시아 거주자한테 Allianz Care 어떤 플랜이 적합한지 추천해주세요.', 'reject_recommend', 'reject_recommend'),
    ('Which plan should I choose for comprehensive outpatient coverage with Allianz?', 'reject_recommend', 'reject_recommend'),
]

#  문맥 이해 — 40개 
# 복합적이거나 문맥 파악이 필요한 질문
context_questions = [
    ('Allianz Care에서 입원 사전승인을 받았는데 치료 계획이 바뀌면 다시 신청해야 하나요?', 'rag', 'context'),
    ('I was admitted as an emergency but Allianz later found it was not urgent. Will my claim be rejected?', 'rag', 'context'),
    ('Allianz Care에서 연간 한도를 다 썼는데 이후 치료는 어떻게 되나요?', 'rag', 'context'),
    ('두 나라에 걸쳐 치료를 받았어요. Allianz에서 어느 나라 기준으로 보장하나요?', 'rag', 'context'),
    ('Allianz Care 보험 갱신 전에 받은 치료인데 갱신 후 청구하면 어떻게 되나요?', 'rag', 'context'),
    ('I received treatment in a non-network hospital. Will Allianz reimburse at a lower rate?', 'rag', 'context'),
    ('Allianz Care에서 치료비 일부만 환급된 이유가 무엇인가요?', 'rag', 'context'),
    ('사전승인 없이 긴급 수술을 받았는데 Allianz에서 나중에 승인을 소급 적용해주나요?', 'rag', 'context'),
    ('Allianz Care에서 동일 질환으로 두 나라에서 치료를 받았을 때 한도는 합산되나요?', 'rag', 'context'),
    ('My Allianz claim was partially denied due to "not medically necessary". How do I appeal?', 'rag', 'context'),
    ('Allianz Care에서 가족 구성원이 각각 독립적으로 연간 한도를 적용받나요?', 'rag', 'context'),
    ('임신 중 Allianz Care에 가입했는데 출산 관련 보장이 적용되나요?', 'rag', 'context'),
    ('Allianz Care 플랜을 변경했는데 기존에 받던 치료가 새 플랜에서 커버가 안 되면 어떻게 되나요?', 'rag', 'context'),
    ('I have two insurance policies including Allianz Care. Which pays first?', 'rag', 'context'),
    ('Allianz Care에서 만성 질환 치료를 매년 갱신할 때 대기 기간이 다시 적용되나요?', 'rag', 'context'),
    ('치료받은 나라의 영수증이 현지 언어(예: 아랍어)인데 Allianz 청구에 문제없나요?', 'rag', 'context'),
    ('Allianz Care에서 응급 치료 후 후속 외래 진료도 같은 건으로 청구 가능한가요?', 'rag', 'context'),
    ('I was referred by my GP for a specialist. Does Allianz Care still require pre-authorisation?', 'rag', 'context'),
    ('Allianz Care에서 치료비 선지불 후 환급받을 때 환율은 어떻게 적용되나요?', 'rag', 'context'),
    ('My child needs long-term physiotherapy. How many sessions per year does Allianz cover?', 'rag', 'context'),
    ('Allianz Care에서 치료 중 보험을 해지하면 진행 중인 청구는 어떻게 되나요?', 'rag', 'context'),
    ('I was hospitalized in a country not listed in my area of cover. Will Allianz pay?', 'rag', 'context'),
    ('Allianz Care에서 외래 치료 한도와 입원 치료 한도가 서로 공유되나요?', 'rag', 'context'),
    ('사전승인 신청했는데 Allianz에서 거절했어요. 이의제기 방법은 무엇인가요?', 'rag', 'context'),
    ('Allianz Care에서 치료 전 진단 검사 비용도 보장이 되나요?', 'rag', 'context'),
    ('My doctor recommended an experimental drug. Will Allianz Care cover it?', 'rag', 'context'),
    ('Allianz Care에서 같은 질환으로 두 번 입원했을 때 한도가 합산되나요?', 'rag', 'context'),
    ('I had surgery abroad. How do I submit the claim if the invoice is in a foreign language?', 'rag', 'context'),
    ('Allianz Care에서 보험 적용 지역 밖에서 응급 치료를 받으면 보장이 되나요?', 'rag', 'context'),
    ('사전승인 받은 치료인데 실제 비용이 승인 금액보다 초과됐어요. 어떻게 되나요?', 'rag', 'context'),
    ('Allianz Care에서 요양원(Nursing home) 비용은 보장이 되나요?', 'rag', 'context'),
    ('My claim was approved but payment was delayed. What should I do?', 'rag', 'context'),
    ('Allianz Care에서 코로나19 관련 치료는 보장이 되나요?', 'rag', 'context'),
    ('I have a pre-existing condition that was excluded. Can I apply for coverage after remission?', 'rag', 'context'),
    ('Allianz Care에서 정신건강 치료와 일반 외래 치료 한도는 별도로 적용되나요?', 'rag', 'context'),
    ('My employer pays for Allianz Care. If I leave the job, can I continue the policy?', 'rag', 'context'),
    ('Allianz Care에서 입원 중 발생한 동반자 숙박비도 보장이 되나요?', 'rag', 'context'),
    ('I received treatment in the US but my area of cover does not include the US. What happens?', 'rag', 'context'),
    ('Allianz Care에서 치료를 위한 의료 항공편(Medical evacuation) 비용은 보장되나요?', 'rag', 'context'),
    ('신생아가 NICU에 입원했어요. Allianz Care에서 어떻게 보장하나요?', 'rag', 'context'),
]

#  개인정보 차단 — 140개 
# 식별 가능한 개인정보(보험번호, 청구번호 등) 포함 → 차단
# 진단명·증상만 있는 경우 → 차단 X, 일반 답변
privacy_block_questions = [
    # 차단 O: 식별 가능 정보 포함 (70개)
    ('제 보험번호 AL-123456이고 당뇨 진단인데 Allianz에서 커버해주나요?', 'reject_privacy', 'reject_privacy'),
    ('My policy number is AL-987654. Can you confirm my coverage for cancer treatment?', 'reject_privacy', 'reject_privacy'),
    ('제 청구 번호 CLM-AL-2024-001로 처리 상황 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('My claim reference is CLM-2024-SGP-789. When will it be paid?', 'reject_privacy', 'reject_privacy'),
    ('보험번호 AL-77777랑 이름 알려드릴게요. 보장 내역 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My member ID is MEM-AL-2024-0001. Please verify my benefits.', 'reject_privacy', 'reject_privacy'),
    ('제 생년월일이랑 보험번호 드릴게요 — 1985-06-15, AL-55555. 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My date of birth is 1990-01-01 and policy number AL-33333. Is my plan active?', 'reject_privacy', 'reject_privacy'),
    ('청구 레퍼런스 CLM-AL-HK-2024-001 진행 상황 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can you look up my Allianz claim history using my member number AL-11111?', 'reject_privacy', 'reject_privacy'),
    ('제 이름이랑 정책 번호 드릴 테니 커버리지 상세히 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('My full name is Sarah Lee and policy ID is AL-CARE-2024-001.', 'reject_privacy', 'reject_privacy'),
    ('사전승인 번호 AUTH-AL-2024-999가 아직 유효한가요?', 'reject_privacy', 'reject_privacy'),
    ('My authorisation number is AUTH-2024-UK-123. Is it still valid?', 'reject_privacy', 'reject_privacy'),
    ('제 계좌번호 KR-123-456-789로 환급금 보내주세요.', 'reject_privacy', 'reject_privacy'),
    ('Please transfer my reimbursement to IBAN DE89370400440532013000.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 AL-44444이고 조현병 진단이에요. 이 조합으로 커버 확인돼요?', 'reject_privacy', 'reject_privacy'),
    ('My policy AL-22222 and I have a diagnosis of F20. Is inpatient covered?', 'reject_privacy', 'reject_privacy'),
    ('제 이메일 abc@test.com이고 보험번호 AL-99999예요. 로그인 도와주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can you help reset my Allianz portal login? My username is jdoe2024.', 'reject_privacy', 'reject_privacy'),
    ('제 주소랑 보험번호 드릴게요 — 서울 용산구, AL-66666. 자격 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My address is 123 Main St, London and policy is AL-88888. Verify my coverage.', 'reject_privacy', 'reject_privacy'),
    ('청구 번호 CLM-AL-2024-5555 처리가 됐는지 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('Has my claim CLM-AL-2024-7890 been approved yet?', 'reject_privacy', 'reject_privacy'),
    ('제 진료 기록 번호 MR-AL-001로 보장 여부 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My patient file number at the hospital is PN-AL-2024-456. Check my benefits.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 AL-11223이고 불안장애 진단이에요. 이 조합으로 커버 확인돼요?', 'reject_privacy', 'reject_privacy'),
    ('Policy AL-33445 — I have F41.1. Is outpatient therapy covered?', 'reject_privacy', 'reject_privacy'),
    ('제 스폰서 ID AL-SPONSOR-001로 가족 보장 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My sponsor policy ID is AL-FAM-2024-001. Confirm my dependents\' coverage.', 'reject_privacy', 'reject_privacy'),
    ('보험사 측에 제 케이스 번호 CASE-AL-2024-9999를 조회해달라고 할 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My case number with Allianz is CASE-2024-HK-001. What is the status?', 'reject_privacy', 'reject_privacy'),
    ('제 Allianz 회원 ID로 지난 청구 내역 조회해줄 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('Can you pull my Allianz claims history using member ID AL-MEM-5678?', 'reject_privacy', 'reject_privacy'),
    ('보험번호 AL-00001이고 HIV 양성이에요. 이 조합으로 보장 확인돼요?', 'reject_privacy', 'reject_privacy'),
    ('Policy AL-11111 — I am HIV positive. Is antiretroviral therapy covered?', 'reject_privacy', 'reject_privacy'),
    ('제 생년월일, 이름, 보험번호 드릴게요. 전체 보장 내역 출력해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My name, DOB, and policy number: Alice Kim, 1988-03-20, AL-CARE-88.', 'reject_privacy', 'reject_privacy'),
    ('청구 레퍼런스 CLM-AL-VN-2024-001 처리 현황 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('What is the status of my overseas claim CLM-AL-OCONUS-2024-001?', 'reject_privacy', 'reject_privacy'),
    ('이름이랑 생년월일, 보험번호 제공하면 보장 내역 다 알 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('Can I provide my name and policy number for a coverage verification?', 'reject_privacy', 'reject_privacy'),
    ('제 Allianz 가입일이랑 보험번호 알려드릴게요. 지금도 유효한지 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can you verify if policy AL-99001 is still active using my date of birth?', 'reject_privacy', 'reject_privacy'),
    ('병원 영수증 번호 INV-AL-2024-KR-001이에요. 이걸로 청구 조회 가능한가요?', 'reject_privacy', 'reject_privacy'),
    ('My invoice reference is INV-AL-2024-001. Can you check if the claim was submitted?', 'reject_privacy', 'reject_privacy'),
    ('제 진단 코드 F33.0이고 보험번호 AL-11223입니다. 사전승인 처리해주세요.', 'reject_privacy', 'reject_privacy'),
    ('Policy AL-55544 with diagnosis F33.0 — please process pre-authorisation.', 'reject_privacy', 'reject_privacy'),
    ('제 케이스 매니저한테 연락해달라고 할 수 있어요? 제 번호는 010-9999-8888이에요.', 'reject_privacy', 'reject_privacy'),
    ('Please have my case manager call me at +44-7700-900001.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 AL-77788이고 전화번호 010-1234-5678이에요. 커버리지 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('Policy AL-66699 — call me at +65-9876-5432 to confirm my benefits.', 'reject_privacy', 'reject_privacy'),
    ('제 EOB 참조번호 EOB-AL-2024-7654로 청구 내역 설명해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My EOB reference EOB-AL-2024-001 — can you explain the charges?', 'reject_privacy', 'reject_privacy'),
    ('이름이 홍길동이고 생년월일 1990-03-15, 보험번호 AL-99999예요. 혜택 확인해줘요.', 'reject_privacy', 'reject_privacy'),
    ('My full name is John Smith, DOB 1975-07-04, policy AL-12345. Confirm benefits.', 'reject_privacy', 'reject_privacy'),
    ('제 입원 청구가 승인됐는지 보험번호 AL-12345로 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('Is my inpatient claim under policy AL-67890 approved?', 'reject_privacy', 'reject_privacy'),
    ('제 청구 내역 확인하려면 보험번호 드리면 돼요? AL-22334입니다.', 'reject_privacy', 'reject_privacy'),
    ('Can I give you my policy number AL-55511 to check claim status?', 'reject_privacy', 'reject_privacy'),
    ('보험번호랑 주소 드릴게요. 제 청구 처리 현황 알 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My policy AL-98765 and I live at 45 Orchard Rd, Singapore. Confirm coverage.', 'reject_privacy', 'reject_privacy'),
    ('제 Allianz 계정 비밀번호 재설정 좀 해주세요. 이메일은 xyz@test.com이에요.', 'reject_privacy', 'reject_privacy'),
    ('Can you help me reset my Allianz account? Email: test@example.com.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 AL-00777이고 주민번호 앞 6자리는 850315예요. 확인해줘요.', 'reject_privacy', 'reject_privacy'),
    ('Policy AL-33311 and my passport number is A12345678. Verify eligibility.', 'reject_privacy', 'reject_privacy'),
    ('제 청구 레퍼런스 CLM-AL-2024-5555로 처리 현황 알 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('Can you confirm if claim AUTH-AL-2024-777 was pre-authorized?', 'reject_privacy', 'reject_privacy'),
    ('제 이름, 생년월일, 보험번호 드릴게요. 전체 보장 내역 출력해주세요.', 'reject_privacy', 'reject_privacy'),
    ('I\'d like to provide my policy ID and passport number for coverage verification.', 'reject_privacy', 'reject_privacy'),
    ('My patient number at the clinic is PN-AL-00987. Pull my Allianz benefits.', 'reject_privacy', 'reject_privacy'),
    ('When will claim CLM-AL-OCONUS-9999 be paid? It\'s overdue.', 'reject_privacy', 'reject_privacy'),

    # 차단 X: 진단명·증상만 있고 식별 정보 없음 → 일반 정보 답변 (70개)
    ('저는 우울증인데 Allianz Care에서 정신건강 치료를 받을 수 있나요?', 'rag', 'context_health_info'),
    ('내가 당뇨 진단을 받았는데 Allianz에서 커버되나요?', 'rag', 'context_health_info'),
    ('불안장애가 있는데 Allianz Care에서 상담 치료 받을 수 있나요?', 'rag', 'context_health_info'),
    ('I have bipolar disorder. Does Allianz Care cover my medication?', 'rag', 'context_health_info'),
    ('조현병 진단인데 Allianz Care 입원 치료 커버되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with major depressive disorder. What Allianz mental health benefits do I have?', 'rag', 'context_health_info'),
    ('아이가 자폐 진단을 받았는데 Allianz Care에서 치료 커버해주나요?', 'rag', 'context_health_info'),
    ('My child has ADHD. Does Allianz Care cover behavioral therapy?', 'rag', 'context_health_info'),
    ('알코올 의존증인데 Allianz Care에서 치료 프로그램 보장이 되나요?', 'rag', 'context_health_info'),
    ('I struggle with substance use. What Allianz Care benefits apply?', 'rag', 'context_health_info'),
    ('섭식 장애 있는데 Allianz Care에서 치료 커버 돼요?', 'rag', 'context_health_info'),
    ('My teenager has an eating disorder. Is residential treatment covered by Allianz?', 'rag', 'context_health_info'),
    ('공황장애 있는데 Allianz Care 약 처방 커버되나요?', 'rag', 'context_health_info'),
    ('I have generalized anxiety disorder. How does Allianz Care cover therapy?', 'rag', 'context_health_info'),
    ('산후 우울증인데 Allianz Care에서 상담 치료 보장이 돼요?', 'rag', 'context_health_info'),
    ('My spouse has PTSD. What mental health services does Allianz Cover?', 'rag', 'context_health_info'),
    ('당뇨 진단인데 Allianz Care에서 인슐린 처방 커버되나요?', 'rag', 'context_health_info'),
    ('I have type 2 diabetes. Are my medications covered under Allianz Care?', 'rag', 'context_health_info'),
    ('고혈압 약 Allianz Care에서 커버돼요?', 'rag', 'context_health_info'),
    ('I was recently diagnosed with hypertension. What Allianz pharmacy benefits apply?', 'rag', 'context_health_info'),
    ('암 진단을 받았어요. Allianz Care에서 항암 치료 커버가 어떻게 되나요?', 'rag', 'context_health_info'),
    ('I have been diagnosed with breast cancer. What Allianz Care benefits do I have?', 'rag', 'context_health_info'),
    ('심장 수술이 필요한데 Allianz Care에서 커버되나요?', 'rag', 'context_health_info'),
    ('I need knee replacement surgery. How does Allianz Care cover this?', 'rag', 'context_health_info'),
    ('임신 중인데 Allianz Care에서 산전 검사는 무료인가요?', 'rag', 'context_health_info'),
    ('I\'m pregnant. What maternity services does Allianz Care cover?', 'rag', 'context_health_info'),
    ('천식 있는데 Allianz Care에서 흡입기 처방 커버돼요?', 'rag', 'context_health_info'),
    ('My child has asthma. Are nebulizer treatments covered by Allianz Care?', 'rag', 'context_health_info'),
    ('뇌졸중 후 재활 치료 Allianz Care에서 보장돼요?', 'rag', 'context_health_info'),
    ('I had a stroke. What rehabilitation services are covered under Allianz Care?', 'rag', 'context_health_info'),
    ('알레르기 면역 치료 Allianz Care에서 보장돼요?', 'rag', 'context_health_info'),
    ('I have severe allergies. Does Allianz Care cover allergy shots?', 'rag', 'context_health_info'),
    ('갑상선 질환 치료 Allianz Care에서 커버되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with hypothyroidism. Are my medications covered by Allianz?', 'rag', 'context_health_info'),
    ('류마티스 관절염 치료 약 Allianz Care에서 커버돼요?', 'rag', 'context_health_info'),
    ('I have rheumatoid arthritis. What specialty drugs does Allianz Care cover?', 'rag', 'context_health_info'),
    ('청력 손실이 있는데 Allianz Care에서 보청기 커버되나요?', 'rag', 'context_health_info'),
    ('I have hearing loss. Does Allianz Care cover hearing aids?', 'rag', 'context_health_info'),
    ('수술 후 물리치료 Allianz Care에서 몇 회까지 커버돼요?', 'rag', 'context_health_info'),
    ('I need post-surgery physical therapy. How many sessions does Allianz Care cover?', 'rag', 'context_health_info'),
    ('다발성 경화증(MS) 치료 Allianz Care에서 보장돼요?', 'rag', 'context_health_info'),
    ('I have multiple sclerosis. What does Allianz Care cover for my condition?', 'rag', 'context_health_info'),
    ('크론병 치료 Allianz Care에서 보장돼요?', 'rag', 'context_health_info'),
    ('I have Crohn\'s disease. What treatments does Allianz Care cover?', 'rag', 'context_health_info'),
    ('루푸스 진단인데 Allianz Care에서 어떤 치료가 커버되나요?', 'rag', 'context_health_info'),
    ('I have lupus. What Allianz Care benefits apply to autoimmune treatment?', 'rag', 'context_health_info'),
    ('HIV 양성인데 Allianz Care에서 항바이러스 약 커버되나요?', 'rag', 'context_health_info'),
    ('I am HIV positive. Does Allianz Care cover antiretroviral therapy?', 'rag', 'context_health_info'),
    ('파킨슨병 진단인데 Allianz Care에서 어떤 치료가 보장되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with Parkinson\'s disease. What Allianz Care benefits apply?', 'rag', 'context_health_info'),
    ('치매 진단인데 Allianz Care에서 어떤 지원이 되나요?', 'rag', 'context_health_info'),
    ('My parent has dementia. What Allianz Care services are available?', 'rag', 'context_health_info'),
    ('간질(뇌전증) 치료 Allianz Care에서 커버돼요?', 'rag', 'context_health_info'),
    ('I have epilepsy. Are anti-seizure medications covered under Allianz Care?', 'rag', 'context_health_info'),
    ('만성 신장 질환인데 Allianz Care에서 투석 치료 커버되나요?', 'rag', 'context_health_info'),
    ('I have kidney disease and need dialysis. Is this covered under Allianz Care?', 'rag', 'context_health_info'),
    ('허리 디스크 진단인데 Allianz Care에서 MRI 커버되나요?', 'rag', 'context_health_info'),
    ('I have a herniated disc. Does Allianz Care cover MRI and specialist visits?', 'rag', 'context_health_info'),
    ('자궁내막증 치료 Allianz Care에서 보장되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with endometriosis. What treatments does Allianz Care cover?', 'rag', 'context_health_info'),
    ('척추 측만증 교정 수술 Allianz Care에서 커버되나요?', 'rag', 'context_health_info'),
    ('My child has scoliosis. Does Allianz Care cover corrective surgery?', 'rag', 'context_health_info'),
    ('대장암 진단인데 Allianz Care에서 수술이랑 항암 치료 커버되나요?', 'rag', 'context_health_info'),
    ('I have colon cancer. What does Allianz Care cover for chemotherapy?', 'rag', 'context_health_info'),
    ('수술 후 감염으로 재입원했는데 Allianz Care에서 추가 입원도 커버되나요?', 'rag', 'context_health_info'),
    ('I was readmitted due to post-surgery infection. Does Allianz cover the second admission?', 'rag', 'context_health_info'),
    ('골다공증 치료 Allianz Care에서 보장되나요?', 'rag', 'context_health_info'),
    ('I have osteoporosis. Are bone density treatments covered by Allianz Care?', 'rag', 'context_health_info'),
    ('건선(Psoriasis) 치료 Allianz Care에서 커버돼요?', 'rag', 'context_health_info'),
    ('I have psoriasis. Does Allianz Care cover biologic treatments?', 'rag', 'context_health_info'),
]

# 전체 질문 목록 조합
# 수량 조절 시 각 리스트 뒤에 슬라이싱 추가
# 예: 400→200개로 줄이려면:
#   QUESTIONS = disclaimer_questions[:70] + reject_recommend_questions[:40] + context_questions[:20] + privacy_block_questions[:70]
# 400개 생성: QUESTIONS = disclaimer_questions + reject_recommend_questions + context_questions + privacy_block_questions
QUESTIONS = disclaimer_questions[:35] + reject_recommend_questions[:20] + context_questions[:10] + privacy_block_questions[:35]

print(f'총 질문 수: {len(QUESTIONS)}개')
from collections import Counter
cats = Counter(cat for _, _, cat in QUESTIONS)
for cat, cnt in sorted(cats.items()):
    print(f'  {cat}: {cnt}개')

총 질문 수: 100개
  context: 10개
  disclaimer_coverage: 30개
  disclaimer_regional: 5개
  reject_privacy: 35개
  reject_recommend: 20개


---
## Cell 2 · 거절 패턴 템플릿 정의

In [13]:
# 거절 패턴 템플릿

REJECT_RECOMMEND_KO = (
    "본 서비스는 특정 Allianz Care 플랜을 추천하거나 가입을 권유하지 않습니다.\n\n"
    "각 플랜(Care Base, Care Enhanced, Care Signature)의 보장 범위(Coverage), "
    "연간 한도(Annual limit), 공동부담금(Co-payment) 등 객관적인 조건을 비교해 드릴 수 있습니다. "
    "어떤 플랜들의 조건을 비교해 드릴까요?\n\n"
    "⚠️ 최종 플랜 선택은 본인의 상황과 보장 필요를 고려하여 "
    "Allianz Care 공식 담당자 또는 공인 보험 설계사와 직접 상담하시기 바랍니다."
)

REJECT_RECOMMEND_EN = (
    "This service does not recommend or endorse any specific Allianz Care plan.\n\n"
    "I can provide objective information on coverage, annual limits, and co-payments "
    "for Care Base, Care Enhanced, and Care Signature so you can make an informed decision. "
    "Which plans would you like to compare?\n\n"
    "⚠️ For final plan selection, please consult directly with an Allianz Care representative "
    "or a licensed insurance advisor."
)

# 개인 식별 정보 포함 시 차단 (보험번호, 청구번호, 여권번호 등)
REJECT_PRIVACY_KO = (
    "개인 식별 정보(보험번호, 청구번호, 여권번호, 계좌번호 등)는 "
    "이 서비스에서 필요하지 않으며 저장되지 않습니다.\n\n"
    "본 서비스는 공개된 Allianz Care 문서를 기반으로 플랜 레벨의 보장 정보를 안내합니다. "
    "이용 중인 Allianz Care 플랜명과 보장 지역을 알려주시면, 해당 플랜의 일반적인 보장 범위와 "
    "한도 기준을 안내해 드릴 수 있습니다.\n\n"
    "⚠️ 개인 계약별 적용 여부는 반드시 Allianz Care 공식 채널을 통해 확인하세요."
)

REJECT_PRIVACY_EN = (
    "Personal identifying information such as policy numbers, claim IDs, passport numbers, "
    "or account details are not required and are not stored by this service.\n\n"
    "This service provides plan-level coverage information based on official Allianz Care documents. "
    "Please share your Allianz Care plan name and region, and I can provide general coverage "
    "and benefit limit information.\n\n"
    "⚠️ For individual claim or eligibility inquiries, please contact Allianz Care directly."
)

def get_reject_answer(question: str, reject_type: str) -> str:
    lang = detect_language(question)
    if reject_type == 'reject_recommend':
        return REJECT_RECOMMEND_KO if lang == 'ko' else REJECT_RECOMMEND_EN
    elif reject_type == 'reject_privacy':
        return REJECT_PRIVACY_KO if lang == 'ko' else REJECT_PRIVACY_EN
    return '해당 질문에 답변할 수 없습니다.'

print('✅ 거절 패턴 템플릿 정의 완료')

✅ 거절 패턴 템플릿 정의 완료


---
## Cell 3 · 데이터셋 생성 (메인 루프)

In [14]:
import json
import time

from rag_utils import run_chat_turn

SYSTEM_PROMPT = (
    "당신은 Allianz Care 국제 건강보험 전문 안내 assistant입니다. "
    "글로벌 및 지역별 수혜자를 주요 대상으로 합니다. "
    "공개된 Allianz Care 공식 문서 기반으로만 플랜 정보를 제공하며, "
    "특정 플랜 추천·가입 권유는 하지 않습니다. "
    "개인 식별 정보(보험번호·청구번호·여권번호 등)는 요청하거나 저장하지 않습니다. "
    "모든 PDF 출처는 '[출처]: 파일명 연도 p.N' 형식으로 명시합니다. "
    "보험 전문 용어는 한국어 답변 시 원어 병기 형식(예: 연간 한도(Annual limit))으로 표기합니다."
)

dataset = []
errors  = []

for idx, (question, q_type, category) in enumerate(QUESTIONS):
    try:
        if q_type in ('reject_recommend', 'reject_privacy'):
            answer = get_reject_answer(question, q_type)
            sources = []

        else:  # rag
            # 수정: generate_answer() 대신 run_chat_turn() 직접 호출
            # generate_answer() 내부에서 thread_id 인자 충돌이 발생하므로
            # run_chat_turn()에 빈 conversation_state로 직접 호출하는 방식으로 변경
            result = run_chat_turn(
                question=question,
                conversation_state={}
            )
            answer = result.get('answer', '')
            docs   = result.get('retrieved_docs', [])
            sources = [
                {
                    'source':   d.metadata.get('source', 'unknown'),
                    'page':     d.metadata.get('page', ''),
                    'region':   d.metadata.get('region', ''),
                    'doc_year': d.metadata.get('doc_year', ''),
                }
                for d in docs
            ]

        dataset.append({
            'messages': [
                {'role': 'system',    'content': SYSTEM_PROMPT},
                {'role': 'user',      'content': question},
                {'role': 'assistant', 'content': answer},
            ],
            'metadata': {
                'category': category,
                'type':     q_type,
                'sources':  sources,
            }
        })

        if (idx + 1) % 10 == 0:
            print(f'[{idx+1}/{len(QUESTIONS)}] 완료 — 최근 질문: {question[:40]}...')

        time.sleep(0.5)  # API rate limit 방지

    except Exception as e:
        errors.append({'idx': idx, 'question': question, 'error': str(e)})
        print(f'  ⚠️ [{idx}] 오류: {question[:40]} → {e}')
        continue

print(f'\n✅ 생성 완료: {len(dataset)}개 / 오류: {len(errors)}개')

!!! Data: {'language': 'ko', 'intent': 'coverage', 'region': 'global', 'english_query': 'How is inpatient treatment covered by Allianz Care?', 'keywords': ['Allianz Care', 'inpatient treatment', 'coverage']}


C:\SKN_24\3차 프로젝트(팀)\src\allianz\rag_utils.py:120: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


Doc: DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf             | dense_rank: 1 |                 bm25_rank: 7 |                     dense_rrf: 0.0164 |                         bm25_rrf: 0.0149
관련성 점수 Scoring doc DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf (region: global, type: benefit_guide) => score: 8
Doc: care-tob-en_보장금액.pdf             | dense_rank: 2 |                 bm25_rank: None |                     dense_rrf: 0.0161 |                         bm25_rrf: 0.0000
관련성 점수 Scoring doc care-tob-en_보장금액.pdf (region: global, type: tob) => score: 7
Doc: care-tob-en_보장금액.pdf             | dense_rank: 2 |                 bm25_rank: None |                     dense_rrf: 0.0161 |                         bm25_rrf: 0.0000
관련성 점수 Scoring doc care-tob-en_보장금액.pdf (region: global, type: tob) => score: 7
Doc: care-tob-en_보장금액.pdf             | dense_rank: 4 |                 bm25_rank: None |                     dense_rrf: 0.0156 |                         bm25_rrf: 0.0000
관련성 점수 Scoring doc care-tob-en_보장금액.pdf

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\kjw70\miniconda3\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kjw70\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

!!! Data: {'language': 'en', 'intent': 'coverage', 'region': 'global', 'english_query': 'What does Allianz Care cover for outpatient medical treatment?', 'keywords': ['Allianz Care', 'coverage', 'outpatient medical treatment']}
Doc: DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf             | dense_rank: 1 |                 bm25_rank: 1 |                     dense_rrf: 0.0164 |                         bm25_rrf: 0.0164
관련성 점수 Scoring doc DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf (region: global, type: benefit_guide) => score: 8
Doc: care-tob-en_보장금액.pdf             | dense_rank: 1 |                 bm25_rank: 4 |                     dense_rrf: 0.0164 |                         bm25_rrf: 0.0156
관련성 점수 Scoring doc care-tob-en_보장금액.pdf (region: global, type: tob) => score: 7
Doc: care-tob-en_보장금액.pdf             | dense_rank: 3 |                 bm25_rank: None |                     dense_rrf: 0.0159 |                         bm25_rrf: 0.0000
관련성 점수 Scoring doc care-tob-en_보장금액.pdf (region: global, type: to

---
## Cell 4 · JSONL 저장

In [15]:
OUTPUT_DIR = './'

# 파인튜닝 메인 파일 (messages만, 학습 시 사용)
main_path = OUTPUT_DIR + 'allianz_finetuning_dataset.jsonl'
with open(main_path, 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps({'messages': item['messages']}, ensure_ascii=False) + '\n')

# 수정: 메타데이터 포함 버전 저장 추가
# messages + metadata(category, type, sources) 를 모두 포함한 파일
# 검증·평가지표 계산 시 사용 (학습에는 사용하지 않음)
meta_path = OUTPUT_DIR + 'allianz_finetuning_dataset_with_meta.jsonl'
with open(meta_path, 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# 오류 목록
if errors:
    err_path = OUTPUT_DIR + 'allianz_finetuning_errors.json'
    with open(err_path, 'w', encoding='utf-8') as f:
        json.dump(errors, f, ensure_ascii=False, indent=2)
    print(f'⚠️ 오류 목록 저장: {err_path}')

print(f'✅ 저장 완료')
print(f'   메인:     {main_path} ({len(dataset)}개)')
# 수정: meta_path 저장 확인 출력 추가
print(f'   메타포함: {meta_path}')

✅ 저장 완료
   메인:     ./allianz_finetuning_dataset.jsonl (100개)
   메타포함: ./allianz_finetuning_dataset_with_meta.jsonl


---
## Cell 5 · 데이터셋 검증

In [16]:
import json
from collections import Counter

# 수정: 검증 대상을 main_path(messages만) 대신 meta_path(메타데이터 포함)로 변경
# sources 필드 포함 여부를 함께 확인하기 위함
loaded = []
with open(meta_path, 'r', encoding='utf-8') as f:
    for line in f:
        loaded.append(json.loads(line.strip()))

print(f'총 데이터 수: {len(loaded)}개\n')

# 카테고리 분포
cats = Counter(item['metadata']['category'] for item in loaded)
print('── 카테고리 분포 ──')
for cat, cnt in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat}: {cnt}개')

# 답변 길이 분포
lengths = [len(item['messages'][2]['content']) for item in loaded]
print(f'\n── 답변 길이 분포 ──')
print(f'  최소: {min(lengths)}자')
print(f'  최대: {max(lengths)}자')
print(f'  평균: {sum(lengths)//len(lengths)}자')

# 짧은 답변 확인 (150자 미만 → 출처 미검색 가능성)
short = [(item['messages'][1]['content'], len(item['messages'][2]['content']))
         for item in loaded if len(item['messages'][2]['content']) < 150]
if short:
    print(f'\n⚠️ 짧은 답변 ({len(short)}개) — 재생성 권장:')
    for q, l in short:
        print(f'  [{l}자] {q[:60]}')
else:
    print('\n✅ 모든 답변 길이 정상')

# 출처 형식 확인
source_ok = sum(1 for item in loaded if '[출처]' in item['messages'][2]['content'])
print(f'\n출처 포함 답변: {source_ok}/{len(loaded)}개')

# 수정: sources 필드 수집 여부 확인 추가
# reject 타입은 sources=[] 이므로 rag 타입 기준으로 집계
has_sources = sum(
    1 for item in loaded
    if item['metadata'].get('sources') and len(item['metadata']['sources']) > 0
)
print(f'sources 수집된 항목: {has_sources}/{len(loaded)}개')

# 샘플 출력 (metadata 포함)
print('\n── 샘플 (첫 번째) ──')
sample = loaded[0]
print(f'Q: {sample["messages"][1]["content"]}')
print(f'A: {sample["messages"][2]["content"][:200]}...')
# 수정: 샘플 출력에 metadata(sources 포함) 추가
print(f'Meta: {sample["metadata"]}')

총 데이터 수: 100개

── 카테고리 분포 ──
  reject_privacy: 35개
  disclaimer_coverage: 30개
  reject_recommend: 20개
  context: 10개
  disclaimer_regional: 5개

── 답변 길이 분포 ──
  최소: 249자
  최대: 2523자
  평균: 854자

✅ 모든 답변 길이 정상

출처 포함 답변: 0/100개
sources 수집된 항목: 45/100개

── 샘플 (첫 번째) ──
Q: Allianz Care에서 입원 치료(Inpatient treatment)는 어떻게 보장되나요?
A: 1. 결론  
Allianz Care는 입원 치료(Inpatient treatment)를 보장하며, 병원에서 의학적으로 입원이 필요한 치료를 대상으로 합니다. 입원 치료 시 처방약과 치료 재료도 보장되며, 사전 승인(pre-authorisation)이 필요할 수 있습니다. 또한, 정부에서 무료로 제공되는 입원 치료의 경우 실제 비용 청구가 없으므로, 입원 ...
Meta: {'category': 'disclaimer_coverage', 'type': 'rag', 'sources': [{'source': 'DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf', 'page': 4, 'region': 'global', 'doc_year': 2025}, {'source': 'DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf', 'page': 57, 'region': 'global', 'doc_year': 2025}, {'source': 'care-tob-en_보장금액.pdf', 'page': 8, 'region': 'global', 'doc_year': 2025}, {'source': 'DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf', 'page': 51, 'region': 'global', 'doc_year': 2025}, {'source': 'c